# TG-119: Code-Based Planning with Machine Learning Outcome Models

![gallery_thumbnail](_static/code_tg_119.png)

## Intro

Welcome to the <i>TG-119: Code-Based Planning with Machine Learning Outcome Models</i> notebook! <br><br> In this notebook, we will augment the <i>TG-119: Conventional Code-Based Planning</i> notebook with machine learning outcome models using data from the TG-119 standard case (available from our [Github repository's docs folder](https://github.com/pyanno4rt/pyanno4rt/tree/master/docs) as .mat- and .csv-files).

> Note: we will focus on the parts beyond conventional planning, and recommend checking out the other notebook if you haven't done yet!

## Import of the relevant classes

In [ ]:
from pyanno4rt.base import (
    Configuration, Evaluation, Optimization, TreatmentPlan)

## Treatment plan initialization

### Setting up the configuration object

In [ ]:
configuration = Configuration(
    label='TG-119-conv',  # Unique identifier for the treatment plan
    modality='photon',  # Treatment modality
    imaging_path='./TG_119_data.mat',  # Path to the CT and segmentation data
    dose_matrix_path='./TG_119_photonDij.mat',  # Path to the dose-influence matrix
    dose_resolution=[6, 6, 6],  # Size of the dose grid in [mm] per dimension
    min_log_level='info',  # Minimum logging level
    number_of_fractions=30  # Number of fractions
    )

### Setting up the optimization object

#### Setting up a machine learning outcome model-based component

We have directly embedded the machine learning outcome models into the optimization components. Therefore, by adding the respective component to the optimization object, these models become a part of the decision process making the solver balance outcome and conventional plan criteria. In the following, we will specify an exemplary logistic regression outcome model-based component step-by-step.

##### Specifying the data path

In [ ]:
data_path = './TG_119_synthetic.csv'

To allow fitting the logistic regression model, we created a synthetic NTCP dataset based on a series of conventional TG-119 plans evaluated with the Lyman-Kutcher-Burman NTCP model to estimate the label values. This data set can be found under the path stated above.

##### Define the input features and the label

In [ ]:
from pyanno4rt.learning.features import DynamicFeature, Label

data_columns = [
    DynamicFeature(column='core_mean', segment='Core', function='Dose Mean'),  # Mean dose
    DynamicFeature(column='core_std', segment='Core', function='Dose Deviation'),  # Standard deviation
    DynamicFeature(column='core_min', segment='Core', function='Dose Minimum'),  # Minimum dose
    DynamicFeature(column='core_max', segment='Core', function='Dose Maximum'),  # Maximum dose
    Label(column='label')  # Binary label for classification
    ]

Dataset handling and feature (re)calculation are essential parts of the model integration. If you provide a dataset, you must therefore define the features and the label using the following classes:

> `pyanno4rt.learning._columns.DynamicFeature`<br>
> <i>represents a "dynamic" feature, i.e., a feature which may be recalculated within each iteration using some function</i>

> `pyanno4rt.learning._columns.StaticFeature`<br>
> <i>represents a "static" feature, i.e., a feature which has a constant, unchanging value</i>

> `pyanno4rt.learning._columns.Label`<br>
> <i>represents the label, which may be binarized and linked to a time after treatment variable</i>

Please check out the <i>Data columns</i> notebook in the [Notebooks](https://pyanno4rt.readthedocs.io/en/latest/notebooks.html) section for more information.

In [ ]:
from pyanno4rt.learning.tune_spaces import TuneSpaceLR

tune_space = TuneSpaceLR(
    C=[2**-5, 2**10],  # 
    penalty=['l1', 'l2', 'elasticnet'],  # 
    tol=[1e-4, 1e-5, 1e-6],  # 
    class_weight=[None, 'balanced']  # 
    )

...

In [ ]:
from pyanno4rt.learning.evaluation import DisplayOptions

display_options = DisplayOptions(
    graphs=['AUC-ROC', 'AUC-PR', 'F1'],  # 
    kpis=[  # 
        'Logloss', 'Brier score', 'Subset accuracy', 'Cohen Kappa', 'Hamming loss',
        'Jaccard score', 'Precision', 'Recall', 'F1 score', 'MCC', 'AUC']
    )

...

In [ ]:
from pyanno4rt.learning import ModelParameters

model_parameters = ModelParameters(
    model_label='lrNTCP',  # 
    model_type='logistic',  # 
    data_path=data_path,  # 
    data_columns=data_columns,  # 
    preprocessing=['StandardScaler'],  # 
    tune_space=tune_space,  # 
    tune_evaluations=50,  # 
    tune_score='AUC',  # 
    tune_splits=5,  # 
    tune_repeats=1,  # 
    inspect=True,  # 
    evaluate=True,  # 
    oof_splits=5,  # 
    oof_repeats=1,  # 
    write_features=True,  # 
    display_options=display_options  # 
    )

...

#### Adding the machine learning outcome model-based optimization component

In [ ]:
from pyanno4rt.optimization.components import (
    LogisticRegressionNTCP, SquaredDeviation, SquaredOverdosing)

optimization = Optimization(
    components=[  # Optimization components for each segment of interest
        LogisticRegressionNTCP(segment='Core', model_parameters=model_parameters, weight=1),
        SquaredOverdosing(segment='Core', maximum_dose=25, weight=100),
        SquaredDeviation(segment='OuterTarget', target_dose=60, weight=1000),
        SquaredOverdosing(segment='BODY', maximum_dose=30, weight=800)],
    method='weighted-sum',  # Single- or multi-criteria optimization method
    solver='scipy',  # Python package to be used for solving the optimization problem
    algorithm='L-BFGS-B',  # Solution algorithm from the chosen solver
    initial_strategy='target-coverage',  # Initialization strategy for the fluence vector
    initial_fluence_vector=None,  # User-defined initial fluence vector (only for 'warm-start')
    lower_variable_bounds=0,  # Lower bounds on the decision variables
    upper_variable_bounds=None,  # Upper bounds on the decision variables
    maximum_iterations=500,  # Maximum number of iterations for the solvers to converge
    tolerance=0.001  # Precision goal for the objective function value
    )

...

> `pyanno4rt.optimization.components._logistic_regression_ntcp.LogisticRegressionNTCP`<br>
> refers to a function that 

### Setting up the evaluation object

In [ ]:
evaluation = Evaluation(
    dvh_type='cumulative',  # Type of DVH to be calculated
    number_of_points=1000,  # Number of (evenly-spaced) points for which to evaluate the DVH
    reference_volume=[2, 5, 50, 95, 98],  # Reference volumes for which to calculate the inverse DVH values
    reference_dose=[],  # Reference dose values for which to calculate the DVH values
    display_segments=[],  # Names of the segmented structures to be displayed
    display_metrics=[]  # Names of the plan evaluation metrics to be displayed
    )

### Initializing the base class

In [ ]:
tp = TreatmentPlan(configuration, optimization, evaluation)

## Treatment plan workflow

...

### Configuring the plan

In [ ]:
tp.configure()

### Modeling for the plan

In [ ]:
tp.model()

...

### Optimizing the plan

In [ ]:
tp.optimize()

### Evaluating the plan

In [ ]:
tp.evaluate()

### Visualizing the plan

In [ ]:
tp.visualize()

...

![pyanno4rt visualizer](_static/visualizer.png)

## Outro

We hope that this little example illustrates the basic usage of the code-based *pyanno4rt* interface for machine learning outcome model-based treatment planning. If you have any remarks, please take a look at the [Help and Support](https://pyanno4rt.readthedocs.io/en/latest/help_support.html) section and drop us a line. We would also be happy if you leave a positive comment and recommend our work to others. <br><br> Thank you for using *pyanno4rt* 😊